# Energy vs Entropy Gating — Comparison Across Architectures

This notebook loads the two trained multi-exit checkpoints from Notebook 1
and evaluates two confidence-based gating mechanisms on both of them.
No training happens here — this is purely evaluation.

**The question being answered:**
> For in-distribution closed-set classification on CIFAR-100, does energy or entropy
> produce better-calibrated early exit decisions, and does the answer differ across
> architectures of different capacity?

| Mechanism | Formula | Input | Range |
|-----------|---------|-------|-------|
| Energy | `−logsumexp(z)` | Raw logits | `(−∞, 0)` |
| Entropy | `−Σ p·log(p)` | Softmax probs | `[0, ln(100)≈4.605]` |

Both use `score < threshold` as the exit condition (lower = more confident).

**Models evaluated:** Multi-Exit ResNet-18 and Multi-Exit ResNet-50

**Evaluations performed per architecture:**
1. Static compute profile (MACs, FLOPs, params, time per exit)
2. Fixed-depth accuracy at each exit (no gating — baseline)
3. Accuracy vs compute curve for both mechanisms
4. Head-to-head at matched compute budgets
5. Calibration: do confident samples have higher accuracy?
6. Score correlation between energy and entropy
7. Final cross-architecture comparison table

## 0 — Imports and device

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import time
from collections import defaultdict

try:
    from thop import profile as thop_profile, clever_format
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','thop','-q'])
    from thop import profile as thop_profile, clever_format

try:
    from fvcore.nn import FlopCountAnalysis
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','fvcore','-q'])
    from fvcore.nn import FlopCountAnalysis

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
Device: cuda
GPU   : Tesla T4


## 1 — Rebuild architectures and load checkpoints

In [3]:
# Shared exit head (must match Notebook 1 exactly)
class ExitHead(nn.Module):
    def __init__(self, in_channels, hidden=512, num_classes=100, dropout=0.3):
        super().__init__()
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(in_channels, hidden)
        self.bn      = nn.BatchNorm1d(hidden)
        self.drop    = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden, num_classes)
    def forward(self, x):
        x = self.flatten(self.pool(x))
        return self.fc2(self.drop(F.relu(self.bn(self.fc1(x)))))


# ResNet-18 
class MultiExitResNet18(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        base = models.resnet18(weights=None)
        self.stem   = nn.Sequential(nn.Conv2d(3,64,3,1,1,bias=False), base.bn1, base.relu)
        self.layer1 = base.layer1; self.layer2 = base.layer2
        self.layer3 = base.layer3; self.layer4 = base.layer4
        self.exit1  = ExitHead( 64,256,num_classes); self.exit2 = ExitHead(128,256,num_classes)
        self.exit3  = ExitHead(256,512,num_classes); self.exit4 = ExitHead(512,512,num_classes)
    def forward(self, x):
        x=self.stem(x); f1=self.layer1(x); f2=self.layer2(f1)
        f3=self.layer3(f2); f4=self.layer4(f3)
        return [self.exit1(f1),self.exit2(f2),self.exit3(f3),self.exit4(f4)]


# ResNet-50 
class MultiExitResNet50(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        base = models.resnet50(weights=None)
        self.stem   = nn.Sequential(nn.Conv2d(3,64,3,1,1,bias=False), base.bn1, base.relu)
        self.layer1 = base.layer1; self.layer2 = base.layer2
        self.layer3 = base.layer3; self.layer4 = base.layer4
        self.exit1  = ExitHead( 256,512,num_classes); self.exit2 = ExitHead( 512,512,num_classes)
        self.exit3  = ExitHead(1024,512,num_classes); self.exit4 = ExitHead(2048,512,num_classes)
    def forward(self, x):
        x=self.stem(x); f1=self.layer1(x); f2=self.layer2(f1)
        f3=self.layer3(f2); f4=self.layer4(f3)
        return [self.exit1(f1),self.exit2(f2),self.exit3(f3),self.exit4(f4)]


# Load two checkpoints
ARCHITECTURES = [
    ('ResNet-18', MultiExitResNet18(), '/kaggle/input/models/pragyansharma2905/n1/pytorch/default/1/best_me_resnet18.pth'),
    ('ResNet-50', MultiExitResNet50(), '/kaggle/input/models/pragyansharma2905/n1/pytorch/default/1/best_me_resnet50.pth'),
]
loaded_models = []
for name, model, ckpt in ARCHITECTURES:
    model = model.to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    p = sum(x.numel() for x in model.parameters())
    loaded_models.append((name, model))
    print(f'  {name:12} loaded from {ckpt}  ({p/1e6:.1f}M params)')

print('\nAll checkpoints loaded.')

  ResNet-18    loaded from /kaggle/input/models/pragyansharma2905/n1/pytorch/default/1/best_me_resnet18.pth  (11.8M params)
  ResNet-50    loaded from /kaggle/input/models/pragyansharma2905/n1/pytorch/default/1/best_me_resnet50.pth  (25.7M params)

All checkpoints loaded.


## 2 — Data loader and gating functions

In [4]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
testset    = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print(f'Test set: {len(testset):,} samples')


# Gating functions
def energy_score(logits):
    """
    E(z) = -logsumexp(z)
    Operates on raw logits before softmax.
    Sensitive to absolute logit magnitude.
    Lower (more negative) = more confident → exit early.
    Originally proposed for OOD detection (Liu et al., 2020).
    """
    return -torch.logsumexp(logits, dim=1)


def entropy_score(logits):
    """
    H(p) = -sum(p * log(p))  where p = softmax(z)
    Scale-invariant — only distribution shape matters.
    Range [0, ln(100)=4.605] for 100 classes.
    Lower = more confident (peaked distribution) → exit early.
    """
    p = torch.softmax(logits, dim=1)
    return -(p * torch.log(p + 1e-8)).sum(dim=1)


GATE_FNS = [
    ('Energy',  energy_score),
    ('Entropy', entropy_score),
]

# Threshold ranges chosen so both span from
# 'almost never exit early' to 'almost always exit at layer 1'
THRESHOLDS = {
    'Energy' : [-9.0,-8.0,-7.0,-6.5,-6.0,-5.5,-5.0,-4.5,-4.0,-3.5,-3.0],
    'Entropy': [ 0.3, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0],
}

print('Data and gating functions ready.')

100%|██████████| 169M/169M [00:02<00:00, 57.3MB/s] 


Test set: 10,000 samples
Data and gating functions ready.


## 3 — Single-pass logit collection

Each model runs once over the full test set. All four exit logits are stored.
Every subsequent comparison (gating simulation, calibration, correlation)
operates on these stored tensors — no additional forward passes needed.
This guarantees both gating mechanisms are compared on identical model outputs.

In [5]:
def collect_logits(model):
    """One forward pass over test set. Returns logits [4][N,100] and labels [N]."""
    all_logits = [[], [], [], []]
    all_labels = []
    model.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device)
            exits  = model(images)
            for i in range(4):
                all_logits[i].append(exits[i].cpu())
            all_labels.append(labels)
    return [torch.cat(l) for l in all_logits], torch.cat(all_labels)


# collect for both models
model_data = {}
for name, model in loaded_models:
    print(f'Collecting logits: {name}...')
    logits, labels = collect_logits(model)
    model_data[name] = {'logits': logits, 'labels': labels}
    # precompute scores
    model_data[name]['energy']  = [energy_score(logits[i])  for i in range(4)]
    model_data[name]['entropy'] = [entropy_score(logits[i]) for i in range(4)]
    acc4 = logits[3].argmax(1).eq(labels).float().mean().item()*100
    print(f'  Full model accuracy (exit 4): {acc4:.2f}%')

print('\nAll logits collected.')

  Full model accuracy (exit 4): 77.56%
  Full model accuracy (exit 4): 81.45%

All logits collected.


## 4 — Static compute profile per architecture

In [7]:
class ExitPrefix(nn.Module):
    """Wraps a ResNet model to run only up to exit k (1-indexed)."""
    def __init__(self, model, exit_k, arch_name):
        super().__init__()
        self.arch   = arch_name
        layers = [model.layer1, model.layer2, model.layer3, model.layer4]
        heads  = [model.exit1,  model.exit2,  model.exit3,  model.exit4]
        self.stem   = model.stem
        self.layers = nn.ModuleList(layers[:exit_k])
        self.head   = heads[exit_k - 1]

    def forward(self, x):
        x = self.stem(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(x)


def fmt(n):
    if n >= 1e9:  return f'{n/1e9:.3f}G'
    if n >= 1e6:  return f'{n/1e6:.3f}M'
    if n >= 1e3:  return f'{n/1e3:.3f}K'
    return f'{n:.3f}'


WARMUP = 30
dummy  = torch.randn(1,3,32,32).to(device)

mac_profiles  = {}
time_profiles = {}

for name, model in loaded_models:
    print(f'\n--- {name} ---')
    print(f"{'Exit':>6} {'MACs':>14} {'FLOPs':>14} {'Params':>12} {'µs/sample':>12}")
    print('-' * 62)
    macs_list = []; times_list = []

    for k in range(1, 5):
        prefix = ExitPrefix(model, k, name).to(device)
        prefix.eval()

        with torch.no_grad():
            macs, params = thop_profile(prefix, inputs=(dummy,), verbose=False)
        fc = FlopCountAnalysis(prefix, dummy)
        fc.unsupported_ops_warnings(False)
        flops = fc.total()
        macs_list.append(macs)

        total_t = 0.0; total_s = 0; batches = 0
        with torch.no_grad():
            for imgs, _ in testloader:
                imgs = imgs.to(device)
                if batches < WARMUP: prefix(imgs); batches+=1; continue
                if device.type=='cuda': torch.cuda.synchronize()
                t0 = time.perf_counter(); prefix(imgs)
                if device.type=='cuda': torch.cuda.synchronize()
                total_t += time.perf_counter()-t0; total_s += imgs.size(0); batches+=1
        us = (total_t/total_s)*1e6
        times_list.append(us)

        print(f"  Exit {k}  {fmt(macs):>12}  {fmt(flops):>12}  {fmt(params):>10}  {us:>10.2f}µs")

    full_mac = macs_list[-1]
    full_t   = times_list[-1]
    mac_profiles[name]  = macs_list
    time_profiles[name] = times_list

    print('  MAC%:', ['%.1f%%'%(100*m/full_mac) for m in macs_list])
    print('  Speedup vs full:', ['%.2f×'%(full_t/t) for t in times_list])


--- ResNet-18 ---
  Exit           MACs          FLOPs       Params    µs/sample
--------------------------------------------------------------
  Exit 1      154.184M      153.528M    192.676K       83.73µs
  Exit 2      289.041M      288.057M    734.628K      125.62µs
  Exit 3      423.695M      422.546M      2.959M      161.35µs
  Exit 4      558.199M      556.969M     11.484M      200.60µs
  MAC%: ['27.6%', '51.8%', '75.9%', '100.0%']
  Speedup vs full: ['2.40×', '1.60×', '1.24×', '1.00×']

--- ResNet-50 ---
  Exit           MACs          FLOPs       Params    µs/sample
--------------------------------------------------------------
  Exit 1      226.349M      223.333M    401.572K      218.87µs
  Exit 2      565.957M      560.909M      1.752M      434.15µs
  Exit 3        1.047G        1.041G      9.113M      668.56µs
  Exit 4        1.313G        1.306G     24.602M      812.07µs
  MAC%: ['17.2%', '43.1%', '79.8%', '100.0%']
  Speedup vs full: ['3.71×', '1.87×', '1.21×', '1.00×']


## 5 — Gating simulation

For each architecture and each gating mechanism, we simulate per-sample routing
across all thresholds using the pre-collected logits.
A sample exits at the first layer where its score is below the threshold.
The final layer is always the fallback.

In [8]:
def simulate_gating(scores_per_exit, logits_per_exit, labels, thresholds, mac_list):
    N = labels.size(0)
    full_macs = mac_list[-1]
    results   = []

    for tau in thresholds:
        exit_layer  = torch.full((N,), 4, dtype=torch.long)
        pred_logits = logits_per_exit[3].clone()
        still_running = torch.ones(N, dtype=torch.bool)

        for i in range(3):   # check exits 1, 2, 3; exit 4 is fallback
            confident     = still_running & (scores_per_exit[i] < tau)
            exit_layer[confident]  = i + 1
            pred_logits[confident] = logits_per_exit[i][confident]
            still_running = still_running & ~confident

        preds   = pred_logits.argmax(1)
        acc     = preds.eq(labels).float().mean().item() * 100
        ef      = [(exit_layer==k+1).sum().item()/N for k in range(4)]
        eff_mac = sum(ef[i]*mac_list[i] for i in range(4))
        mac_save= 100.0*(1 - eff_mac/full_macs)
        avg_exit= exit_layer.float().mean().item()

        results.append({
            'tau':tau,'acc':acc,'avg_exit':avg_exit,
            'exit_frac':ef,'mac_save':mac_save,'eff_mac':eff_mac
        })
    return results


# run simulation for all architectures × both mechanisms
all_results = {}

for name, model in loaded_models:
    d = model_data[name]
    all_results[name] = {}
    for gate_name, gate_fn in GATE_FNS:
        scores = d[gate_name.lower()]
        res = simulate_gating(
            scores, d['logits'], d['labels'],
            THRESHOLDS[gate_name], mac_profiles[name]
        )
        all_results[name][gate_name] = res

print('Simulation complete for both architectures × 2 gating mechanisms.')

Simulation complete for both architectures × 2 gating mechanisms.


## 6 — Fixed-depth accuracy baseline (no gating)

In [9]:
fixed_acc = {}   # fixed_acc[arch_name][exit_k] = accuracy

for name, _ in loaded_models:
    d = model_data[name]
    fixed_acc[name] = {}
    print(f'\n{name} — fixed-depth accuracy')
    print(f"{'Exit':>6} {'Accuracy':>10} {'MAC%':>8}")
    print('-' * 30)
    full_mac = mac_profiles[name][-1]
    for k in range(4):
        acc = d['logits'][k].argmax(1).eq(d['labels']).float().mean().item()*100
        pct = 100.0*mac_profiles[name][k]/full_mac
        fixed_acc[name][k+1] = acc
        print(f"  Exit {k+1}  {acc:>8.2f}%  {pct:>6.1f}%")


ResNet-18 — fixed-depth accuracy
  Exit   Accuracy     MAC%
------------------------------
  Exit 1     31.82%    27.6%
  Exit 2     64.89%    51.8%
  Exit 3     77.16%    75.9%
  Exit 4     77.56%   100.0%

ResNet-50 — fixed-depth accuracy
  Exit   Accuracy     MAC%
------------------------------
  Exit 1     18.17%    17.2%
  Exit 2     69.07%    43.1%
  Exit 3     81.59%    79.8%
  Exit 4     81.45%   100.0%


## 7 — Head-to-head at matched compute budgets

For each architecture and each compute budget (10–60% MAC saving),
find the closest operating point for each mechanism and compare accuracy directly.

In [10]:
def closest(results, target_save):
    return min(results, key=lambda r: abs(r['mac_save']-target_save))


BUDGETS = [10, 20, 30, 40, 50, 60]
head2head_summary = {}   # store for final table

for name, _ in loaded_models:
    full_acc_val = fixed_acc[name][4]
    en_res = all_results[name]['Energy']
    et_res = all_results[name]['Entropy']

    print(f'\n{name}  (full model: {full_acc_val:.2f}%)')
    print(f"{'Budget':>8} {'Energy acc':>12} {'Entropy acc':>13} {'Δ':>8} {'Winner':>10}")
    print('-' * 58)

    wins = {'Energy':0,'Entropy':0,'Tie':0}
    rows = []
    for b in BUDGETS:
        en = closest(en_res, b); et = closest(et_res, b)
        delta  = et['acc'] - en['acc']
        winner = 'Entropy' if delta>0.1 else 'Energy' if delta<-0.1 else 'Tie'
        wins[winner] += 1
        rows.append({'budget':b,'en_acc':en['acc'],'et_acc':et['acc'],'winner':winner})
        print(f"  {b:>3}% save  {en['acc']:>10.2f}%  {et['acc']:>11.2f}%  "
              f"{delta:>+7.2f}%  {winner:>10}")

    print(f'  Energy wins: {wins["Energy"]}  Entropy wins: {wins["Entropy"]}  Ties: {wins["Tie"]}')
    head2head_summary[name] = {'rows':rows,'wins':wins,'full_acc':full_acc_val}


ResNet-18  (full model: 77.56%)
  Budget   Energy acc   Entropy acc        Δ     Winner
----------------------------------------------------------
   10% save       77.03%        77.44%    +0.41%     Entropy
   20% save       76.44%        76.73%    +0.29%     Entropy
   30% save       72.47%        74.70%    +2.23%     Entropy
   40% save       72.47%        71.84%    -0.63%      Energy
   50% save       62.94%        65.78%    +2.84%     Entropy
   60% save       62.94%        65.78%    +2.84%     Entropy
  Energy wins: 1  Entropy wins: 5  Ties: 0

ResNet-50  (full model: 81.45%)
  Budget   Energy acc   Entropy acc        Δ     Winner
----------------------------------------------------------
   10% save       81.12%        81.35%    +0.23%     Entropy
   20% save       80.61%        81.11%    +0.50%     Entropy
   30% save       79.30%        79.85%    +0.55%     Entropy
   40% save       75.49%        78.39%    +2.90%     Entropy
   50% save       75.49%        75.97%    +0.48%   

## 8 — Calibration analysis

A well-calibrated gate separates confident samples (low score) from uncertain ones.
We bin samples into deciles by score and check if accuracy decreases monotonically
from the most confident decile to the least confident.
Monotonicity score = fraction of adjacent decile pairs with the expected ordering.

In [11]:
def monotonicity(bins):
    pairs = [(bins[i]['acc'] >= bins[i+1]['acc']) for i in range(len(bins)-1)]
    return sum(pairs)/len(pairs)*100


def calibration_deciles(scores, logits, labels, n=10):
    preds   = logits.argmax(1)
    correct = preds.eq(labels)
    order   = scores.argsort()
    sc_s    = scores[order]; co_s = correct[order]
    N       = len(scores); bs = N//n
    bins = []
    for b in range(n):
        s = b*bs; e = (b+1)*bs if b<n-1 else N
        bins.append({'decile':b+1,'mean_score':sc_s[s:e].mean().item(),'acc':co_s[s:e].float().mean().item()*100})
    return bins


print('CALIBRATION ANALYSIS — monotonicity scores (100% = perfect calibration)\n')
print(f"{'Arch':>12} {'Exit':>6} {'Energy mono%':>14} {'Entropy mono%':>15}")
print('-' * 52)

calib_results = {}
for name, _ in loaded_models:
    d = model_data[name]
    calib_results[name] = {'energy':[],'entropy':[]}
    for exit_idx in range(4):
        en_bins = calibration_deciles(d['energy'][exit_idx],  d['logits'][exit_idx], d['labels'])
        et_bins = calibration_deciles(d['entropy'][exit_idx], d['logits'][exit_idx], d['labels'])
        em = monotonicity(en_bins); etm = monotonicity(et_bins)
        calib_results[name]['energy'].append(em)
        calib_results[name]['entropy'].append(etm)
        print(f"  {name:>10}  Exit {exit_idx+1}  {em:>12.0f}%  {etm:>13.0f}%")
    # average across exits
    avg_en = sum(calib_results[name]['energy'])/4
    avg_et = sum(calib_results[name]['entropy'])/4
    print(f"  {'':>10}  avg    {avg_en:>12.0f}%  {avg_et:>13.0f}%")
    print()

CALIBRATION ANALYSIS — monotonicity scores (100% = perfect calibration)

        Arch   Exit   Energy mono%   Entropy mono%
----------------------------------------------------
   ResNet-18  Exit 1           100%            100%
   ResNet-18  Exit 2           100%            100%
   ResNet-18  Exit 3           100%            100%
   ResNet-18  Exit 4            89%            100%
              avg              97%            100%

   ResNet-50  Exit 1            89%             89%
   ResNet-50  Exit 2           100%            100%
   ResNet-50  Exit 3           100%             89%
   ResNet-50  Exit 4           100%            100%
              avg              97%             94%



## 9 — Score correlation

How much do energy and entropy agree on which samples are confident?
Spearman rank correlation tells us whether they produce the same *ordering*
of samples from most to least confident, regardless of scale differences.

In [12]:
def spearman(x, y):
    def rank(t):
        o = t.argsort(); r = torch.zeros_like(o, dtype=torch.float)
        r[o] = torch.arange(len(t), dtype=torch.float); return r
    rx=rank(x)-rank(x).mean(); ry=rank(y)-rank(y).mean()
    return (rx*ry).sum()/(rx.norm()*ry.norm()+1e-8)


print('Spearman rank correlation between energy and entropy scores\n')
print(f"{'Arch':>12} {'Exit1':>8} {'Exit2':>8} {'Exit3':>8} {'Exit4':>8} {'Interpretation':>22}")
print('-' * 72)

for name, _ in loaded_models:
    d = model_data[name]
    corrs = [spearman(d['energy'][i], d['entropy'][i]).item() for i in range(4)]
    avg   = sum(corrs)/4
    interp = 'Very strong' if avg>0.9 else 'Strong' if avg>0.7 else 'Moderate' if avg>0.5 else 'Weak'
    print(f"  {name:>10}  {'  '.join(f'{c:>6.3f}' for c in corrs)}  {interp:>22}")

Spearman rank correlation between energy and entropy scores

        Arch    Exit1    Exit2    Exit3    Exit4         Interpretation
------------------------------------------------------------------------
   ResNet-18   0.995   0.986   0.983   0.940             Very strong
   ResNet-50   0.999   0.992   0.983   0.915             Very strong


## 10 — Correct vs incorrect score distributions

The gate only saves compute if confident samples are actually correct.
Separation = (mean score of wrong predictions) − (mean score of correct predictions).
Larger separation = gate more reliably distinguishes correct from incorrect.

In [13]:
print('Score separation: wrong mean − correct mean at each exit')
print('(larger = gate better separates correct from incorrect predictions)\n')
print(f"{'Arch':>12} {'Gate':>10} {'Exit1':>8} {'Exit2':>8} {'Exit3':>8} {'Exit4':>8}")
print('-' * 62)

separation_results = {}
for name, _ in loaded_models:
    d = model_data[name]
    separation_results[name] = {'energy':[],'entropy':[]}
    for gate_name, gate_key in [('Energy','energy'),('Entropy','entropy')]:
        seps = []
        for i in range(4):
            scores  = d[gate_key][i]
            correct = d['logits'][i].argmax(1).eq(d['labels'])
            sep = (scores[~correct].mean() - scores[correct].mean()).item()
            seps.append(sep)
        separation_results[name][gate_key] = seps
        print(f"  {name:>10}  {gate_name:>10}  {'  '.join(f'{s:>6.3f}' for s in seps)}")
    print()

Score separation: wrong mean − correct mean at each exit
(larger = gate better separates correct from incorrect predictions)

        Arch       Gate    Exit1    Exit2    Exit3    Exit4
--------------------------------------------------------------
   ResNet-18      Energy   0.137   0.798   0.882   0.377
   ResNet-18     Entropy   0.203   1.053   1.334   0.910

   ResNet-50      Energy   0.068   0.707   0.563   0.263
   ResNet-50     Entropy   0.105   1.073   1.180   0.713



## 11 - Exit Distribution

In [15]:
print('EXIT DISTRIBUTION — images out of 10,000 at each layer\n')

for name in ['ResNet-18', 'ResNet-50']:
    for gate_name in ['Energy', 'Entropy']:
        res = all_results[name][gate_name]
        thresholds = THRESHOLDS[gate_name]

        print(f'{name} — {gate_name}')
        print(f"{'τ':>6} {'Exit1':>8} {'Exit2':>8} {'Exit3':>8} {'Exit4':>8} {'MACsave':>9} {'Acc':>7}")
        print('-' * 65)

        for r in res:
            ef = r['exit_frac']
            print(f"  {r['tau']:>4.1f}  "
                  f"{int(ef[0]*10000):>7}  "
                  f"{int(ef[1]*10000):>7}  "
                  f"{int(ef[2]*10000):>7}  "
                  f"{int(ef[3]*10000):>7}  "
                  f"{r['mac_save']:>7.1f}%  "
                  f"{r['acc']:>5.2f}%")
        print()

EXIT DISTRIBUTION — images out of 10,000 at each layer

ResNet-18 — Energy
     τ    Exit1    Exit2    Exit3    Exit4   MACsave     Acc
-----------------------------------------------------------------
  -9.0        0      187      161     9652      1.3%  77.56%
  -8.0        0      584      566     8850      4.2%  77.47%
  -7.0        2     1606     1492     6899     11.4%  77.03%
  -6.5       11     2478     2008     5502     16.9%  76.44%
  -6.0       78     3706     2266     3950     23.9%  75.32%
  -5.5      402     5359     2070     2169     33.7%  72.47%
  -5.0     2060     6727      874      339     49.5%  62.94%
  -4.5    10000        0        0        0     72.4%  31.82%
  -4.0    10000        0        0        0     72.4%  31.82%
  -3.5    10000        0        0        0     72.4%  31.82%
  -3.0    10000        0        0        0     72.4%  31.82%

ResNet-18 — Entropy
     τ    Exit1    Exit2    Exit3    Exit4   MACsave     Acc
---------------------------------------------

## 12 — Final comparison summary

In [14]:
print('=' * 90)
print('ENERGY vs ENTROPY GATING — CROSS-ARCHITECTURE SUMMARY')
print('=' * 90)

print('\n[1] FULL MODEL ACCURACY (exit 4, no gating)')
print(f"{'Architecture':>16} {'Accuracy':>10} {'Params':>10}")
print('-' * 40)
for name, model in loaded_models:
    p = sum(x.numel() for x in model.parameters())
    print(f"  {name:>14}  {fixed_acc[name][4]:>8.2f}%  {p/1e6:>8.1f}M")

print('\n[2] HEAD-TO-HEAD WINS ACROSS COMPUTE BUDGETS (10–60% MAC saving)')
print(f"{'Architecture':>16} {'Energy wins':>13} {'Entropy wins':>14} {'Ties':>6} {'Overall winner':>16}")
print('-' * 72)
for name in [n for n,_ in loaded_models]:
    w = head2head_summary[name]['wins']
    overall = 'Energy' if w['Energy']>w['Entropy'] else 'Entropy' if w['Entropy']>w['Energy'] else 'Tie'
    print(f"  {name:>14}  {w['Energy']:>11}  {w['Entropy']:>12}  {w['Tie']:>4}  {overall:>16}")

print('\n[3] AVERAGE CALIBRATION MONOTONICITY (higher = better calibrated)')
print(f"{'Architecture':>16} {'Energy avg%':>13} {'Entropy avg%':>14} {'Better':>10}")
print('-' * 56)
for name in [n for n,_ in loaded_models]:
    avg_en = sum(calib_results[name]['energy'])/4
    avg_et = sum(calib_results[name]['entropy'])/4
    better = 'Energy' if avg_en>avg_et else 'Entropy'
    print(f"  {name:>14}  {avg_en:>11.1f}%  {avg_et:>12.1f}%  {better:>10}")

print('\n[4] AVERAGE SCORE SEPARATION AT FINAL EXIT (higher = gate more informative)')
print(f"{'Architecture':>16} {'Energy sep':>12} {'Entropy sep':>13} {'Better':>10}")
print('-' * 56)
for name in [n for n,_ in loaded_models]:
    en_sep = separation_results[name]['energy'][-1]
    et_sep = separation_results[name]['entropy'][-1]
    better = 'Energy' if abs(en_sep)>abs(et_sep) else 'Entropy'
    print(f"  {name:>14}  {en_sep:>10.3f}  {et_sep:>11.4f}  {better:>10}")

print('\n[5] RECOMMENDED OPERATING POINTS (~30% MAC saving)')
print(f"{'Arch':>12} {'Gate':>10} {'Accuracy':>10} {'MAC save':>10} {'AccDrop':>9}")
print('-' * 58)
for name in [n for n,_ in loaded_models]:
    full_a = fixed_acc[name][4]
    for gate_name in ['Energy','Entropy']:
        best = min(all_results[name][gate_name], key=lambda r:abs(r['mac_save']-30))
        drop = full_a - best['acc']
        print(f"  {name:>10}  {gate_name:>10}  {best['acc']:>8.2f}%  {best['mac_save']:>8.1f}%  {drop:>+8.2f}%")
    print()

print('=' * 90)
print('INTERPRETATION')
print('=' * 90)
print('''
Energy operates on raw logit magnitudes before softmax.
Entropy operates on the shape of the softmax probability distribution.

If entropy wins consistently across architectures:
  → Scale-invariance matters. The softmax distribution shape is a more
    reliable confidence signal than raw logit magnitudes for in-distribution
    classification on CIFAR-100.

If energy wins or results are mixed:
  → Logit magnitude is itself an informative signal, and removing it via
    softmax normalisation discards useful information. The two mechanisms
    are measuring partially different aspects of uncertainty.

If both mechanisms agree (high Spearman correlation):
  → The choice of gate matters less than the choice of threshold.
    Practical differences will be small and dataset-specific.
''')

ENERGY vs ENTROPY GATING — CROSS-ARCHITECTURE SUMMARY

[1] FULL MODEL ACCURACY (exit 4, no gating)
    Architecture   Accuracy     Params
----------------------------------------
       ResNet-18     77.56%      11.8M
       ResNet-50     81.45%      25.7M

[2] HEAD-TO-HEAD WINS ACROSS COMPUTE BUDGETS (10–60% MAC saving)
    Architecture   Energy wins   Entropy wins   Ties   Overall winner
------------------------------------------------------------------------
       ResNet-18            1             5     0           Entropy
       ResNet-50            0             6     0           Entropy

[3] AVERAGE CALIBRATION MONOTONICITY (higher = better calibrated)
    Architecture   Energy avg%   Entropy avg%     Better
--------------------------------------------------------
       ResNet-18         97.2%         100.0%     Entropy
       ResNet-50         97.2%          94.4%      Energy

[4] AVERAGE SCORE SEPARATION AT FINAL EXIT (higher = gate more informative)
    Architecture   Energ